# 4.3. The Base Classification Model
D2L의 The Base Classification Model장을 PyTorch 기준으로 정리함.

- 모델 예측 class 구하기
- accuracy 계산
- validation 데이터 평가
- optimizer 설정

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import time

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 분류 모델 출력

분류 모델은 정답 class 번호 하나를 바로 출력하지 않음.
각 class에 대한 점수를 출력한다.
예를 들어 class가 3개면 모델 출력 하나는 이럴수 있다.

    [0.1, 0.7, 0.2]

- class 0 점수: 0.1
- class 1 점수: 0.7
- class 2 점수: 0.2

가장 높은 점수는 0.7이라 모델은 class 1을 예측한다.

In [ ]:
Y_hat = torch.tensor([
    [0.1, 0.7, 0.2],
    [2.0, 1.0, 0.0],
    [0.2, 0.3, 0.5],
    [0.9, 0.05, 0.05]
])

Y = torch.tensor([1, 0, 2, 2])

print("Y_hat shape:", Y_hat.shape) # [batch_size, num_classes]
print("Y shape:", Y.shape) # 각 데이터 정답 class 번호

Y_hat shape: torch.Size([4, 3])
Y shape: torch.Size([4])
tensor([1, 0, 2, 2])


## 2. argmax로 예측 class 구하기

`argmax()`는 가장 큰 값의 위치를 반환한다.
첫 번째 모델 출력은 다음과 같다.
    [0.1, 0.7, 0.2]
    
가장 큰 값인 0.7은 index 1에 있다. 그래서 argmax() = 1이 나온다. 모델은 class 1을 예측한다.

In [6]:
print(Y_hat[0])
print(Y_hat[0].argmax())

tensor([0.1000, 0.7000, 0.2000])
tensor(1)


`argmax(dim=1)`은 각 행에서 가장 큰 값의 위치를 찾는다.

    [0.1, 0.7, 0.2] → 1
    [2.0, 1.0, 0.0] → 0
    [0.2, 0.3, 0.5] → 2
    [0.9, 0.05, 0.05] → 0
    
따라서 모델의 예측은 다음과 같다.

    Y_pred = [1, 0, 2, 0]

In [7]:
Y_pred = Y_hat.argmax(dim=1)
print(Y_pred)

tensor([1, 0, 2, 0])


## 3. Accuracy 계산

Accuracy는 전체 데이터 중 모델이 정답을 맞힌 비율이다.

$$ 
\text{accuracy} = \frac{\text{맞힌 데이터 개수}} {\text{전체 데이터 개수}} $$

예측값과 정답을 비교하면 각 데이터가 맞았는지 확인할 수 있다.

In [8]:
print("예측:", Y_pred) 
print("정답:", Y) 

correct = Y_pred == Y 
print("비교:", correct)

예측: tensor([1, 0, 2, 0])
정답: tensor([1, 0, 2, 2])
비교: tensor([ True,  True,  True, False])


예측과 정답을 비교하면 Boolean Tensor가 나온다.

In [ ]:
correct_float = correct.float() # Boolean을 float으로 변환
# True -> 1.0 , False -> 0.0

print(correct_float)
print(correct_float.mean()) # accuracy 는 75%

tensor([1., 1., 1., 0.])
tensor(0.7500)


## 4. Accuracy 함수 만들기

`accuracy()` 함수는 다음 순서로 동작한다.

1. `argmax(dim=1)`로 예측 class를 구한다. 
2. 예측값과 정답을 비교한다. 
3. True와 False를 1과 0으로 변환한다. 
4. 평균을 계산한다.

전체 흐름은 이렇다.

    Y_hat
    → argmax
    → Y_pred
    → 정답 Y와 비교
    → 평균
    → accuracy

In [10]:
def accuracy(Y_hat, Y):
    """
    분류 모델의 accuracy를 계산한다.
    Y_hat shape: [batch_size, num_classes]
    Y shape: [batch_size]
    """
    
    Y_pred = Y_hat.argmax(dim=1)
    correct = (Y_pred == Y).float()
    
    return correct.mean()

acc = accuracy(Y_hat, Y)

print("accuracy:", acc) 
print("accuracy 값:", acc.item())

accuracy: tensor(0.7500)
accuracy 값: 0.75


## 5. Softmax 없이 argmax 사용?

모델이 출력하는 softmax 이전의 점수는 logits이다.

예측 class만 구할 때는 logits에 softmax를 적용하지 않고 바로 `argmax()`를 사용해도 된다.

Softmax는 점수를 확률 형태로 바꾸지만, 가장 큰 값의 위치는 바꾸지 않기 때문이다.

In [11]:
logits = torch.tensor([
    [2.0, 5.0, 1.0]
])

probabilities = torch.softmax(logits, dim=1)

print("logits:", logits) 
print("probabilities:", probabilities)

print("logits argmax:", logits.argmax(dim=1)) 
print("softmax argmax:", probabilities.argmax(dim=1))

logits: tensor([[2., 5., 1.]])
probabilities: tensor([[0.0466, 0.9362, 0.0171]])
logits argmax: tensor([1])
softmax argmax: tensor([1])


두 argmax 결과는 똑같다. 1 1

예측 class만 필요하다면 이렇게 사용해도 된다

    Y_pred = logits.argmax(dim=1)

확률값을 직접 확인 할 때만 softmax를 적용하면 된다.

## 6. Loss와 Accuracy 차이

Loss와 accuracy는 서로 다른 역할을 한다.
### Loss

모델이 얼마나 잘못 예측했는지를 나타낸다.

모델은 loss를 미분해서 gradient를 계산하고, 파라미터를 업데이트한다.

    loss.backward() optimizer.step()
    
### Accuracy

전체 데이터 중 몇 개를 맞혔는지 나타내는 평가 지표이다.
    accuracy = 맞힌 개수 / 전체 개수

Loss -> 모델 학습에 사용
Accuracy -> 모델 성능 확인에 사용

예를 들어서 정답 class가 1이라고 하자.

두 모델 출력이 다음과 같을 수 있다.

    예측 A = [0.01, 0.51, 0.48] 예측 B = [0.01, 0.98, 0.01]

두 예측 모두 가장 큰 값이 class 1에 있으므로 accuracy 관점에서는 둘 다 정답이다.

근데 예측 B가 정답 class에 훨씬 높은 확신을 보인다.

Accuracy는 이 차이를 구분하지 못하지만, 
cross-entropy loss는 이 차이를 구분할 수 있다.

그래서 모델은 accuracy가 아니라 loss를 기준으로 학습한다.

## 7. Validation Accuracy 계산

실제 validation에서는 minibatch 하나가 아니라 DataLoader에 있는 전체 데이터를 평가해야 한다.
전체 accuracy는 다음과 같이 계산한다.

    전체 맞힌 개수 / 전체 데이터 개수

In [14]:
@torch.no_grad() # gardient 계산 막기
def evaluate_accuracy(model, data_loader, device="cpu"):
    model.eval() # 모드변경

    correct = 0
    total = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device) # batch 가져오기

        Y_hat = model(X) # Y_hat 계산하기 
        Y_pred = Y_hat.argmax(dim=1)   # argmax로 예측 class 구하기

        correct += (Y_pred == Y).sum().item() # 맞힌 개수 누적
        total += Y.numel()  # 전체 데이터 개수 누적

    return correct / total   # 전체 맞친 개수 전체 데이터개수로 나누기

## 8. Batch Accuracy 평균을 바로 내면 안되는 이유

각 batch의 accuracy를 단순히 더하고 batch 개수로 나누면 마지막 batch가 다를 수 있기 때문에 잘못된 결과가 나올 수 있다.

예를 들어서 

    첫 번째 batch: 64개
    마지막 batch: 8개

두 batch의 accuracy를 같은 비중으로 평균을 내면 데이터가 8개인 batch가 영향을 준다.

    전체 맞힌 개수 / 전체 데이터 개수

In [ ]:
correct += (Y_pred == Y).sum().item() # 현 batch에서 맞힌 개수 계산

total += Y.numel() # 현재 batch의 전체 label 개수 계산

## 9. Optimizer 설정

D2L에서는 기본 optimizer로 SGD를 쓴다.

Optimizer는 gradient를 이용해서 모델의 weight와 bias를 업데이트하는 역할을 한다.

In [ ]:
learning_rate = 0.1

optimizer = torch.optim.SGD(
    model.parameters(), # 업데이터할 모델 weight, bias
    lr=learning_rate # 업데이트 이동 크기
)

## 10. Training, Validation 비교

Training에선 모델 파라키터를 업데이트한다.

```py
model.train()

Y_hat = model(X)
loss = loss_fn(Y_hat, Y) 

optimizer.zero_grad() 
loss.backward() 
optimizer.step()
```

Validation에서는 파라미터를 업데이트하지 않는다.

```py
model.eval() 

with torch.no_grad():
    Y_hat = model(X) 
    loss = loss_fn(Y_hat, Y) 
    acc = accuracy(Y_hat, Y)
```

In [ ]:
model.train()

for X, Y in train_loader:
    optimizer.zero_grad()

    Y_hat = model(X)
    loss = loss_fn(Y_hat, Y) # loss 계산

    loss.backward() # 역전파
    optimizer.step() # 업데이트

model.eval()

with torch.no_grad():
    for X, Y in test_loader:
        Y_hat = model(X) 
        val_loss = loss_fn(Y_hat, Y) # loss 계산
        val_acc = accuracy(Y_hat, Y) # accuracy 계산
        # 역전파, 업데이트 없음

## 11. D2L 코드 해석

D2L에서 분류 모델에서 공통으로 사용할 기능을 `Classifier`라는 부모 클래스에 정의한다. 

원본 코드의 핵심은 이렇다.

    Y_hat = self(*batch[:-1])
    loss = self.loss(Y_hat, batch[-1])
    acc = self.accuracy(Y_hat, batch[-1]) 

이미지 분류에서 batch는 일반적으로 다음 구조다.

    batch = (X, Y)

In [ ]:
batch = (X, Y) 

print(batch[:-1]) 
print(batch[-1])

D2L 코드 PyTorch코드로 바꾸면 이렇게 된다.

In [ ]:
X, Y = batch 

Y_hat = model(X) 

val_loss = loss_fn(Y_hat, Y) 
val_acc = accuracy(Y_hat, Y)

## 13. 오늘의 정리 

- 분류 모델은 각 class에 대한 점수를 출력한다. 
- 모델 출력 `Y_hat`의 shape은 `[batch_size, num_classes]`이다. 
- 정답 `Y`의 shape은 `[batch_size]`이다. 
- `argmax(dim=1)`은 각 데이터의 예측 class를 구한다. 
- Accuracy는 전체 데이터 중 정답을 맞힌 비율이다. 
- Accuracy는 `맞힌 개수 / 전체 데이터 개수`로 계산한다. 
- 모델은 accuracy가 아니라 loss를 사용해서 학습한다. 
- Loss는 예측이 얼마나 잘못되었는지 세밀하게 구분할 수 있다. 
- Training에서는 `backward()`와 `optimizer.step()`을 실행한다. 
- Validation에서는 모델 파라미터를 업데이트하지 않는다. 
- Validation에서는 `model.eval()`과 `torch.no_grad()`를 사용한다. 
- 전체 validation accuracy는 batch별 평균이 아니라 전체 맞힌 개수를 기준으로 계산해야 한다. 
- D2L의 `Classifier`는 분류 모델의 공통 기능을 재사용하기 위한 부모 클래스이다.